# ✅ Aula 10 — Filtragem Colaborativa por Usuário — SOLUÇÃO
### Módulo 13 · Engenharia de Software · Inteli 2026.1B

> **Nota para o professor:** O dataset não está incluído neste notebook.
> Na hora da aula, gere os dados sintéticos com uma LLM e salve como
> `dataset_aula10.csv` nesta mesma pasta — ou cole diretamente na
> célula de carregamento marcada com `# ← INSERIR DADOS AQUI`.

---
Implementações completas de **UserKNN** e **SVD** com avaliação comparativa.


## 📦 0. Dependências e Imports

In [ ]:
# !pip install scikit-surprise pandas numpy matplotlib seaborn scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from math import sqrt
from io import StringIO

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')
print('✅ Imports OK')


---
## 📂 1. Carregamento do Dataset

> Substitua a variável `df` pelo dataset do seu projeto.


In [ ]:
# ← INSERIR DADOS AQUI
# Opção A — arquivo CSV gerado pelo LLM
# df = pd.read_csv('dataset_aula10.csv')

# Opção B — dados inline (para demos rápidas)
# dados_demo = """usuario_id,item_id,rating
# u001,i001,4
# ..."""
# df = pd.read_csv(StringIO(dados_demo))

# ─────────────────────────────────────────────────────
# PLACEHOLDER — substitua pelas linhas acima
df = None
# ─────────────────────────────────────────────────────

assert df is not None, "❌ Carregue o dataset antes de continuar"
print(f'✅ {df.shape[0]} avaliações carregadas')
print(df.head())


---
## 🔍 2. Exploração dos Dados (EDA)


In [ ]:
# Informações gerais
print(df.info())
print('\n', df.describe())
print('\nValores únicos:', df.nunique())


In [ ]:
# Distribuição dos ratings
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['rating'].hist(ax=axes[0], bins=10, edgecolor='white')
axes[0].set_title('Distribuição dos Ratings')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Frequência')

n_usuarios = df['usuario_id'].nunique()
n_itens    = df['item_id'].nunique()
n_ratings  = len(df)
esparsidade = 1 - (n_ratings / (n_usuarios * n_itens))

labels = ['Preenchido', 'Esparso']
sizes  = [n_ratings, n_usuarios * n_itens - n_ratings]
axes[1].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
axes[1].set_title(f'Esparsidade da Matriz\n({esparsidade:.1%} vazia)')

plt.tight_layout()
plt.show()

print(f'Usuários: {n_usuarios} | Itens: {n_itens} | '
      f'Avaliações: {n_ratings} | Esparsidade: {esparsidade:.1%}')


In [ ]:
# Heatmap da matriz usuário-item
matriz_heatmap = df.pivot_table(
    index='usuario_id', columns='item_id', values='rating'
)
plt.figure(figsize=(14, 6))
sns.heatmap(
    matriz_heatmap,
    cmap='YlOrRd',
    linewidths=0.3,
    linecolor='lightgrey',
    annot=len(df['item_id'].unique()) <= 20,  # só anota se couber
    fmt='.0f',
    cbar_kws={'label': 'Rating'}
)
plt.title('Matriz Usuário × Item')
plt.tight_layout()
plt.show()


---
## 🧮 3. Preparação: Matriz Usuário-Item e Split


In [ ]:
# Matriz usuário-item (NaN onde não há avaliação)
matriz = df.pivot_table(
    index='usuario_id', columns='item_id', values='rating'
)
print(f'Matriz: {matriz.shape[0]} usuários × {matriz.shape[1]} itens')
print(matriz.head())


In [ ]:
# Divisão treino/teste (80/20 ao nível de avaliações individuais)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Recria a matriz apenas com os dados de treino
matriz_treino = train_df.pivot_table(
    index='usuario_id', columns='item_id', values='rating'
).reindex(index=matriz.index, columns=matriz.columns)

print(f'Treino: {len(train_df)} avaliações')
print(f'Teste : {len(test_df)} avaliações')


---
## 👥 4. UserKNN — Implementação Completa


In [ ]:
# ── 4.1 Matriz de similaridade (cosine) ────────────────

# Substitui NaN por 0 apenas para o cálculo de similaridade
matriz_filled = matriz_treino.fillna(0).values
sim_matrix    = cosine_similarity(matriz_filled)

sim_df = pd.DataFrame(
    sim_matrix,
    index=matriz_treino.index,
    columns=matriz_treino.index
)

print('Matriz de similaridade:')
print(sim_df.iloc[:5, :5].round(3))


In [ ]:
# ── 4.2 Predição com UserKNN ────────────────────────────

def predict_userknn(usuario, item, matriz, sim_df, k=5):
    """Prevê o rating de (usuario, item) usando UserKNN com k vizinhos."""
    # Verificar se o item existe na matriz
    if item not in matriz.columns:
        return None

    # Usuários que avaliaram este item
    avaliadores = matriz[item].dropna().index.tolist()
    if not avaliadores or usuario not in sim_df.index:
        return None

    # Ordenar vizinhos por similaridade (excluindo o próprio usuário)
    candidatos = [u for u in avaliadores if u != usuario]
    if not candidatos:
        return None

    sims = sim_df.loc[usuario, candidatos].sort_values(ascending=False)
    vizinhos = sims.head(k)

    # Média do usuário-alvo (para ajuste de viés)
    media_alvo = matriz.loc[usuario].mean() if usuario in matriz.index else 0

    numerador   = 0.0
    denominador = 0.0
    for vizinho, sim in vizinhos.items():
        rating_viz  = matriz.loc[vizinho, item]
        media_viz   = matriz.loc[vizinho].mean()
        numerador   += sim * (rating_viz - media_viz)
        denominador += abs(sim)

    if denominador == 0:
        return media_alvo

    return media_alvo + (numerador / denominador)

# Teste rápido
_u = df['usuario_id'].iloc[0]
_i = df['item_id'].iloc[0]
_p = predict_userknn(_u, _i, matriz_treino, sim_df, k=5)
print(f'Predição UserKNN para ({_u}, {_i}): {_p:.2f}' if _p else 'Sem predição')


In [ ]:
# ── 4.3 RMSE no conjunto de teste ───────────────────────

def avaliar_userknn(test_df, matriz, sim_df, k=5):
    y_real, y_pred = [], []
    for _, row in test_df.iterrows():
        pred = predict_userknn(row['usuario_id'], row['item_id'], matriz, sim_df, k)
        if pred is not None:
            y_real.append(row['rating'])
            y_pred.append(pred)
    rmse = sqrt(mean_squared_error(y_real, y_pred))
    return rmse, len(y_pred)

rmse_knn, n_knn = avaliar_userknn(test_df, matriz_treino, sim_df, k=5)
print(f'UserKNN  →  RMSE: {rmse_knn:.4f}  |  Predições: {n_knn}/{len(test_df)}')


In [ ]:
# ── 4.4 Top-N Recomendações com UserKNN ─────────────────

def recomendar_userknn(usuario, matriz, sim_df, k=5, top_n=5):
    """Retorna os top_n itens recomendados, excluindo os já avaliados."""
    if usuario not in matriz.index:
        return []

    ja_avaliados = set(matriz.loc[usuario].dropna().index)
    todos_itens  = set(matriz.columns)
    candidatos   = todos_itens - ja_avaliados

    predicoes = {}
    for item in candidatos:
        pred = predict_userknn(usuario, item, matriz, sim_df, k)
        if pred is not None:
            predicoes[item] = pred

    recomendados = sorted(predicoes.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return pd.DataFrame(recomendados, columns=['item_id', 'predicted_rating'])

usuario_demo = df['usuario_id'].iloc[0]
rec_knn = recomendar_userknn(usuario_demo, matriz_treino, sim_df, k=5, top_n=5)
print(f'\nTop-5 recomendações UserKNN para {usuario_demo}:')
print(rec_knn.to_string(index=False))


---
## 🔢 5. SVD — Implementação Completa


In [ ]:
# ── 5.1 Treinar SVD ──────────────────────────────────────

def treinar_svd(matriz, n_fatores=10):
    """
    Aplica TruncatedSVD sobre a matriz usuário-item e retorna
    a matriz reconstruída (com predições para todos os pares).
    """
    # Preenche NaN com a média de cada usuário (bias correction)
    medias_usuario = matriz.mean(axis=1)
    matriz_norm = matriz.T.fillna(medias_usuario).T

    svd_model = TruncatedSVD(n_components=n_fatores, random_state=42)
    U = svd_model.fit_transform(matriz_norm.values)
    Vt = svd_model.components_

    # Reconstrói a matriz no espaço original
    reconstruida_vals = U @ Vt
    reconstruida = pd.DataFrame(
        reconstruida_vals,
        index=matriz.index,
        columns=matriz.columns
    )

    return reconstruida, svd_model

N_FATORES = min(10, min(matriz_treino.shape) - 1)
matriz_rec, svd_model = treinar_svd(matriz_treino, n_fatores=N_FATORES)

print(f'SVD treinado com {N_FATORES} fatores latentes')
print(f'Variância explicada: {svd_model.explained_variance_ratio_.sum():.1%}')
print(matriz_rec.iloc[:3, :4].round(2))


In [ ]:
# ── 5.2 RMSE no conjunto de teste ───────────────────────

def avaliar_svd(test_df, matriz_reconstruida):
    y_real, y_pred = [], []
    for _, row in test_df.iterrows():
        try:
            pred = matriz_reconstruida.loc[row['usuario_id'], row['item_id']]
            y_real.append(row['rating'])
            y_pred.append(pred)
        except KeyError:
            pass
    rmse = sqrt(mean_squared_error(y_real, y_pred))
    return rmse, len(y_pred)

rmse_svd, n_svd = avaliar_svd(test_df, matriz_rec)
print(f'SVD      →  RMSE: {rmse_svd:.4f}  |  Predições: {n_svd}/{len(test_df)}')


In [ ]:
# ── 5.3 Top-N Recomendações com SVD ─────────────────────

def recomendar_svd(usuario, matriz, matriz_rec, top_n=5):
    """Retorna os top_n itens recomendados via SVD."""
    if usuario not in matriz_rec.index:
        return pd.DataFrame()

    ja_avaliados = set(matriz.loc[usuario].dropna().index) if usuario in matriz.index else set()
    scores = matriz_rec.loc[usuario].drop(labels=list(ja_avaliados), errors='ignore')
    top = scores.nlargest(top_n).reset_index()
    top.columns = ['item_id', 'predicted_rating']
    return top

rec_svd = recomendar_svd(usuario_demo, matriz_treino, matriz_rec, top_n=5)
print(f'\nTop-5 recomendações SVD para {usuario_demo}:')
print(rec_svd.to_string(index=False))


---
## 📊 6. Avaliação Comparativa


In [ ]:
# ── 6.1 Tabela de métricas ──────────────────────────────

resultados = pd.DataFrame({
    'Algoritmo': ['UserKNN (k=5)', f'SVD (d={N_FATORES})'],
    'RMSE': [round(rmse_knn, 4), round(rmse_svd, 4)],
    'Predições Realizadas': [n_knn, n_svd],
    'Coverage (%)': [
        round(100 * n_knn / len(test_df), 1),
        round(100 * n_svd / len(test_df), 1)
    ]
})
print(resultados.to_string(index=False))


In [ ]:
# ── 6.2 Visualizações ───────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Bar chart — RMSE
axes[0].bar(resultados['Algoritmo'], resultados['RMSE'],
            color=['#4C72B0', '#DD8452'], edgecolor='white', linewidth=1.5)
axes[0].set_title('RMSE (menor = melhor)')
axes[0].set_ylabel('RMSE')
for i, v in enumerate(resultados['RMSE']):
    axes[0].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=10)

# Bar chart — Coverage
axes[1].bar(resultados['Algoritmo'], resultados['Coverage (%)'],
            color=['#4C72B0', '#DD8452'], edgecolor='white', linewidth=1.5)
axes[1].set_title('Coverage (%)')
axes[1].set_ylabel('% do teste coberto')
axes[1].set_ylim(0, 110)
for i, v in enumerate(resultados['Coverage (%)']):
    axes[1].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=10)

# Scatter: recomendações em comum
overlap = set(rec_knn['item_id'].tolist()) & set(rec_svd['item_id'].tolist())
somente_knn = set(rec_knn['item_id'].tolist()) - overlap
somente_svd = set(rec_svd['item_id'].tolist()) - overlap

axes[2].barh(['Apenas UserKNN', 'Somente SVD', 'Em comum'],
             [len(somente_knn), len(somente_svd), len(overlap)],
             color=['#4C72B0', '#DD8452', '#55A868'])
axes[2].set_title(f'Overlap Top-5\n(usuário: {usuario_demo})')
axes[2].set_xlabel('Nº de itens')

plt.suptitle('Comparação: UserKNN vs SVD', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ── 6.3 Comparação lado a lado das recomendações ────────

print(f'\n{'='*50}')
print(f'Recomendações para o usuário: {usuario_demo}')
print(f'{'='*50}')

comp = pd.DataFrame({
    'Rank': range(1, 6),
    'UserKNN': rec_knn['item_id'].values if not rec_knn.empty else ['-']*5,
    'Score KNN': rec_knn['predicted_rating'].round(2).values if not rec_knn.empty else ['-']*5,
    'SVD': rec_svd['item_id'].values if not rec_svd.empty else ['-']*5,
    'Score SVD': rec_svd['predicted_rating'].round(2).values if not rec_svd.empty else ['-']*5,
})
print(comp.to_string(index=False))
print(f'\n✅ Itens em comum: {overlap if overlap else "(nenhum)"}')


---
## 🔬 7. Análise dos Fatores Latentes (SVD)


In [ ]:
# Variância explicada por fator
var_exp = svd_model.explained_variance_ratio_

plt.figure(figsize=(9, 4))
plt.bar(range(1, len(var_exp) + 1), var_exp * 100, color='#4C72B0', edgecolor='white')
plt.plot(range(1, len(var_exp) + 1), np.cumsum(var_exp) * 100,
         'o-', color='#DD8452', label='Variância acumulada')
plt.axhline(80, color='grey', linestyle='--', alpha=0.5, label='80%')
plt.xlabel('Fator latente')
plt.ylabel('Variância explicada (%)')
plt.title('Variância explicada por fator — SVD')
plt.legend()
plt.tight_layout()
plt.show()


---
## 📝 8. Gabarito — Pontos do Relatório Comparativo

> Esta seção serve como referência para o professor durante o debrief.
> Os alunos devem preencher o relatório no notebook **template**.

---

### 8.1 Descrição do Dataset
- Domínio, escala, esparsidade, perfis de usuário injetados

### 8.2 Resultados Quantitativos
- Comparar RMSE numérico de ambos os modelos
- Discutir se a diferença é estatisticamente relevante dado o tamanho do dataset

### 8.3 Análise Qualitativa
- As recomendações fazem sentido no domínio?
- Overlap entre os dois modelos para um mesmo usuário

### 8.4 Vantagens e Limitações
**UserKNN:** intuitivo, interpretável, sem fase de treino offline;
mas sofre com usuários frios (cold start) e escala mal com muitos usuários

**SVD:** captura padrões latentes, lida melhor com esparsidade, mais robusto;
mas é menos interpretável e requer retreinamento periódico

### 8.5 Conclusão
- UserKNN → datasets pequenos, domínios onde interpretabilidade importa
- SVD → maior escala, maior esparsidade, produção
- Integração ao projeto: API de recomendação, cold-start handling, atualização incremental
